# Experimental setup

Set up for the experiment. This will load in the required packages, load in the data set, and load in the model. Then the experiment is run on the data set, saving the model weights and the results so they can be reloaded later. Below are the returned metrics.

## Metrics

- Overall performance in learned tasks: Usually average accuracy and average incremental accuracy, and confusion matrix
- Memory stability of old classes: uses forgetting measure (average forgetting of old tasks) and backward transfer (average influence of learning the k-th task on all old tasks)
- Learning plasticity of new classes: intransience measure (inability to learn a new task) and forward transfer (average influence of all old tasks on the current task)
- Resource (storage) and computational overheads (big O complexity), RAM usage.
- Discrepancy of task distribution

## Imports

In [ ]:
# Load packages
import torch
from torch import nn
from torch.utils.data import DataLoader

import torchvision
from torchvision import datasets
from torchvision.transforms import transforms


import numpy as np
import matplotlib.pyplot as plt

import os
import time

print(torch.__version__)
print(torch.cuda.is_available())
torch.set_default_device('cuda')

2.10.0+cu128
True


In [ ]:
# Parameters

LOAD_FROM_FILE = False
FILE_LOAD_PATH = "" + "model_weights.pth"

In [30]:
# Load dataset and model
dataset_path = os.path.join(os.getcwd(), 'datasets')

download = not (os.path.exists(dataset_path))

print('Downloading dataset:',download)

print('Dataset path:', dataset_path)

# Define the desired size for the input tensors
desired_size = (224, 224)

# Create a transformation to resize the input tensors
resize_transform = transforms.Compose([
    transforms.Resize(desired_size),
    transforms.ToTensor()
])

dataset = torchvision.datasets.Imagenette(root = dataset_path, 
                                          split = 'train', 
                                          download = download, transform = resize_transform
                                          )

model = torchvision.models.convnext_tiny()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print(f"Using {device} device")
print(model)


Dataset path: c:\Users\naido\Documents\ChalmersCourses\Thesis\code\datasets
Using cuda device
ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
        

In [ ]:
# Create training metaparameters

# TODO: Probably has minibatches within a class, and each class is separated completely. Either test between each minibatch or between each added class
# Class incremental online learning: Should have the data slowly add in new classes. 
# Then train on the new classes. 
# The old classes should maybe have a few new examples over time? Or just purely the new classes?
train_dataloader = DataLoader(dataset, batch_size=64, shuffle=False)
test_dataloader = DataLoader(dataset, batch_size=64, shuffle=False)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

epochs = 1

def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 10 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")




In [ ]:
# Training
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

timestr = time.strftime("%Y%m%d-%H%M%S")

torch.save(model.state_dict(), timestr + "model_weights.pth")
print("Saved PyTorch Model State to " + timestr + "model_weights.pth")

Epoch 1
-------------------------------
loss: 7.422372  [   64/ 9469]
loss: 0.083191  [ 6464/ 9469]
Test Error: 
 Accuracy: 10.1%, Avg loss: 5.865638 

Done!
Saved PyTorch Model State to model_weights.pth


In [ ]:
# Testing

if LOAD_FROM_FILE:
    model.load_state_dict(torch.load(FILE_LOAD_PATH))

model.eval()
with torch.no_grad():
    correct = 0
    total = 0

    for images, labels in test_dataloader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print('Accuracy of the network test images: %d %%' % (
        100 * correct / total))

Accuracy of the network on the 10000 test images: 10 %


In [ ]:
# Metrics and plots

# TODO: 
# Add in confusion matrix for the classes
# Overall performance in learned tasks: Usually average accuracy and average incremental accuracy
# Memory stability of old classes: uses forgetting measure (average forgetting of old tasks) and backward transfer (average influence of learning the k-th task on all old tasks)
# Learning plasticity of new classes: intransience measure (inability to learn a new task) and forward transfer (average influence of all old tasks on the current task)
# Resource (storage) and computational overheads (big O complexity), RAM usage.
# Discrepancy of task distribution